# When Accuracy Lies

**Notebook 1 of 5.** So far one number has judged every model: accuracy, the share of
predictions that are right. This notebook shows a problem where that number stops meaning
anything: **finding the few fraudulent card payments among many honest ones.**

In [5]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

## 1. The data

Card payments by European card holders over two days. The column `fraud` is 1 for a
fraudulent payment and 0 for an honest one. To protect the card holders, the original
columns were turned into anonymous numbers, `V1` to `V28` (25 of them are included here);
only `amount` is left as it was.

The full dataset has 284,807 payments. This file keeps all 492 frauds and a random sample
of the honest payments, 50,000 rows in all, so that it stays small enough to work with.

In [6]:
df = pd.read_csv("data/transactions.csv")
df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,amount,fraud
0,-1.9873,0.6089,1.6858,0.2671,-0.5133,-0.2206,-0.8481,1.1720,0.1082,-1.2196,...,0.1852,0.2101,-0.2813,0.0601,-0.0237,-0.5414,-0.1617,-0.1671,1.00,0
1,-0.5330,0.7561,2.2770,-0.3065,0.3569,0.2587,0.4523,-0.0951,1.2459,-0.9887,...,-0.2662,-0.3443,-0.3321,-0.4950,0.2732,-0.7228,-0.0827,-0.1588,11.27,0
2,-7.6269,-6.9764,-2.0779,3.4168,4.4588,-5.0804,-6.5789,1.7603,-0.5995,-4.0017,...,1.2248,-0.6566,-0.3308,-0.0789,0.2703,0.4311,0.8214,-1.0561,18.98,1
3,-0.7564,1.6524,-0.7359,-0.7558,0.6905,-0.7352,0.9678,-0.1322,0.7387,0.9326,...,0.2438,1.1961,-0.0632,0.6249,-0.5655,-0.2253,0.9293,0.5667,2.27,0
4,1.0684,-0.2781,1.1889,1.7399,-0.8378,0.5687,-0.6220,0.3632,1.4636,-0.3779,...,-0.4285,-0.8374,0.0968,0.0433,0.3763,-0.5056,0.0811,0.0267,15.99,0


In [7]:
df["fraud"].value_counts()

fraud
0    49508
1      492
Name: count, dtype: int64

**492 frauds, about 1 payment in 100.** When one class is this much rarer than the other,
the data is called **imbalanced**. The rare class, the one we care about, is the
**minority** class; here it is the positive class, 1.

## 2. A model that does nothing

The split first, stratified, as always. With so few frauds, stratifying matters: it
guarantees the test set gets its fair share of them.

In [18]:
X = df.drop(columns="fraud")
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("frauds in the test set:", y_test.sum(), "of", len(y_test))



frauds in the test set: 123 of 12500


The majority baseline: call every payment honest.

In [22]:
y_pred_baseline = np.zeros(len(y_test), dtype=int)   # 0 = honest, for every payment

print(f"accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")

accuracy: 0.9902


**99 percent accurate, and it has never caught a single fraud.** A bank using it would
lose every fraudulent payment, and its accuracy report would look excellent.

## 3. A real model

> **Exercise:**
>
> Scale the features with a `StandardScaler` fitted on the training set, train a
> `LogisticRegression(max_iter=1000)`, and compute its test accuracy. How much better than
> the baseline is it?

<details><summary>
Click here for a hint...
</summary>

The same steps as in the logistic regression repository: `fit_transform` on the training
set, `transform` on the test set, then `fit` and `predict`. `max_iter=1000` only gives the
fitting more steps to finish; the default is too few for 26 columns.
</details>

In [23]:
# your code: scale, train, and compute the test accuracy

# Scale the features using values learned from the training set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create and train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predict the test labels
y_pred = model.predict(X_test_scaled)

# Compare model accuracy with baseline accuracy
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
model_accuracy = accuracy_score(y_test, y_pred)

print(f"Baseline accuracy: {baseline_accuracy:.2%}")
print(f"Model accuracy: {model_accuracy:.2%}")
print(
    f"Improvement: {(model_accuracy - baseline_accuracy) * 100:.2f} "
    "percentage points"
)

Baseline accuracy: 99.02%
Model accuracy: 99.77%
Improvement: 0.75 percentage points


<details><summary>
Click here to compare your answer...
</summary>

About **99.8 percent**, less than one point above doing nothing. Judged by accuracy, the
model has barely earned its keep.
</details>

> **Exercise:**
>
> Accuracy hides the question that matters: **how many of the frauds were caught?** Count,
> for the baseline and for the model, the test frauds that were predicted as fraud.

<details><summary>
Click here for a hint...
</summary>

A fraud is caught where the truth is 1 **and** the prediction is 1:
`((y_test == 1) & (y_pred == 1)).sum()`.
</details>

In [24]:
# your code: frauds caught by the baseline and by the model

# Actual number of fraudulent payments in the test set
total_frauds = (y_test == 1).sum()

# Count payments that were actual frauds AND predicted as fraud
baseline_caught = ((y_test == 1) & (y_pred_baseline == 1)).sum()
model_caught = ((y_test == 1) & (y_pred == 1)).sum()

print(f"Baseline caught: {baseline_caught} of {total_frauds} frauds")
print(f"Model caught: {model_caught} of {total_frauds} frauds")
print(f"Share of frauds caught: {model_caught / total_frauds:.2%}")

Baseline caught: 0 of 123 frauds
Model caught: 97 of 123 frauds
Share of frauds caught: 78.86%


<details><summary>
Click here to compare your answer...
</summary>

The baseline catches **0 of 123**; the model catches **97**. Accuracy put the two within a
point of each other, while one of them is useless and the other catches four frauds in
five. Accuracy counts every payment equally, and 99 in 100 payments are honest, so it
mostly measures how well a model recognizes honest payments.
</details>

## 4. Imbalance is everywhere

Fraud is not an unusual case. The events most worth predicting are often the rare ones:

- **fraud**, among millions of honest payments
- **a rare disease**, among many healthy patients
- **a defective part**, on a production line that mostly makes good ones

The same thing hurts every time. A model learns by getting as many training rows right as
it can, and when 99 rows in 100 belong to one class, ignoring the other class costs almost
nothing. So models lean toward the majority, and accuracy does not notice.

Two things follow, and they are the rest of this repository:

1. **Better metrics**, which look at each class separately: notebooks 02 to 04.
2. **Ways to handle the imbalance** itself: notebook 05.

## Summary

- On **imbalanced** data, a model that always predicts the majority class gets a high
  accuracy and is useless.
- Accuracy counts every row equally, so it is dominated by the majority class.
- The question that matters, **how many of the rare cases were found**, needs a different
  kind of metric.

Next, in `02_confusion_matrix.ipynb`: the table every classification metric is built from.